In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pickle
import re
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import os
import random
import warnings
import fasttext
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090 Ti


In [ ]:
FASTTEXT_MODEL_PATH = ""
train_file = "/blp25/dataset/1A/train.tsv"
validation_file = "/blp25/dataset/1A/dev.tsv"
test_file = "/blp25/dataset/1A/dev_test.tsv"

In [ ]:
train_df = pd.read_csv(train_file, sep='\t')
val_df = pd.read_csv(validation_file, sep='\t')
test_df = pd.read_csv(test_file, sep='\t')

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

print("\nLabel distribution in training data:")
print(train_df['label'].value_counts())

# Handle None values as valid labels
train_df['label'] = train_df['label'].fillna('None')
val_df['label'] = val_df['label'].fillna('None') 

Train samples: 35522
Validation samples: 2512
Test samples: 2512

Label distribution in training data:
label
Abusive           8212
Political Hate    4227
Profane           2331
Religious Hate     676
Sexism             122
Name: count, dtype: int64


In [3]:
def preprocess_text(text):
    """Basic text preprocessing for Bengali text"""
    if pd.isna(text):
        return ""
    
    text = str(text).strip()
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text)
    return text

print("Preprocessing text...")
train_df['text'] = train_df['text'].apply(preprocess_text)
val_df['text'] = val_df['text'].apply(preprocess_text)
test_df['text'] = test_df['text'].apply(preprocess_text)

# Label encoding
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['label'])
val_df['label_encoded'] = label_encoder.transform(val_df['label'])

print(f"\nLabel mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{i}: {label}")

num_classes = len(label_encoder.classes_)

Preprocessing text...

Label mapping:
0: Abusive
1: None
2: Political Hate
3: Profane
4: Religious Hate
5: Sexism


In [4]:
def simple_tokenize(text):
    """Simple whitespace tokenization"""
    return text.lower().split()

def build_vocab(texts, min_freq=2):
    """Build vocabulary from texts"""
    word_freq = {}
    
    for text in texts:
        tokens = simple_tokenize(text)
        for token in tokens:
            word_freq[token] = word_freq.get(token, 0) + 1
    
    # Filter by minimum frequency
    vocab = ['<PAD>', '<UNK>']  # Special tokens
    for word, freq in word_freq.items():
        if freq >= min_freq:
            vocab.append(word)
    
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    
    return word2idx, idx2word, vocab

print("Building vocabulary...")
word2idx, idx2word, vocab = build_vocab(train_df['text'].tolist())
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

def text_to_sequence(text, word2idx, max_length=100):
    """Convert text to sequence of indices"""
    tokens = simple_tokenize(text)
    sequence = []
    
    for token in tokens[:max_length]:
        sequence.append(word2idx.get(token, word2idx['<UNK>']))
    
    return sequence

print("Converting texts to sequences...")
MAX_LENGTH = 100
train_sequences = [text_to_sequence(text, word2idx, MAX_LENGTH) for text in train_df['text']]
val_sequences = [text_to_sequence(text, word2idx, MAX_LENGTH) for text in val_df['text']]
test_sequences = [text_to_sequence(text, word2idx, MAX_LENGTH) for text in test_df['text']]

Building vocabulary...
Vocabulary size: 19785
Converting texts to sequences...


In [ ]:
def load_fasttext_embeddings(fasttext_model_path, word2idx, embedding_dim=300):
    """Load FastText embeddings and create embedding matrix"""
    print(f"Loading FastText embeddings from {fasttext_model_path}...")
    
    if not os.path.exists(fasttext_model_path):
        print(f"Warning: FastText model {fasttext_model_path} not found!")
        raise FileNotFoundError(f"FastText model {fasttext_model_path} not found!")
    
    try:
        if fasttext_model_path.endswith('.bin'):
            ft_model = fasttext.load_model(fasttext_model_path)
            print(f"Loaded binary FastText model with {len(ft_model.words)} words")
        else:
            # Text format (.vec file)
            embeddings_index = {}
            with open(fasttext_model_path, 'r', encoding='utf-8') as f:
                next(f)  # Skip header line
                for line in tqdm(f, desc="Loading FastText"):
                    values = line.split()
                    word = values[0]
                    coefs = np.asarray(values[1:], dtype='float32')
                    embeddings_index[word] = coefs
            print(f"Found {len(embeddings_index)} word vectors in FastText file")
            ft_model = None
            
    except Exception as e:
        print(f"Error loading FastText file: {e}")
        print("Creating random embeddings instead...")
        embedding_matrix = np.random.normal(0, 0.1, (len(word2idx), embedding_dim))
        return embedding_matrix
    
    embedding_matrix = np.zeros((len(word2idx), embedding_dim))
    found_words = 0
    
    for word, idx in tqdm(word2idx.items(), desc="Creating embedding matrix"):
        try:
            if ft_model is not None:  # Binary model
                embedding_vector = ft_model.get_word_vector(word)
                embedding_matrix[idx] = embedding_vector
                found_words += 1
            else:  # Text format
                embedding_vector = embeddings_index.get(word)
                if embedding_vector is not None:
                    embedding_matrix[idx] = embedding_vector
                    found_words += 1
                else:
                    # Initialize with random values for OOV words
                    embedding_matrix[idx] = np.random.normal(0, 0.1, embedding_dim)
        except:
            # Initialize with random values if error occurs
            embedding_matrix[idx] = np.random.normal(0, 0.1, embedding_dim)
    
    print(f"Found embeddings for {found_words}/{len(word2idx)} words ({found_words/len(word2idx)*100:.2f}%)")
    return embedding_matrix

EMBEDDING_DIM = 300
embedding_matrix = load_fasttext_embeddings(FASTTEXT_MODEL_PATH, word2idx, EMBEDDING_DIM)

Loading FastText embeddings from /home/nafi/dev/shared-task/blp25/paper-codes/embeddings/cc.bn.300.vec...


Loading FastText: 1468578it [00:39, 36987.16it/s]
Loading FastText: 1468578it [00:39, 36987.16it/s]


Found 1468578 word vectors in FastText file


Creating embedding matrix: 100%|██████████| 19785/19785 [00:00<00:00, 827336.22it/s]

Found embeddings for 18821/19785 words (95.13%)


In [6]:
class HateSpeechDataset(Dataset):
    def __init__(self, sequences, labels=None):
        self.sequences = sequences
        self.labels = labels
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            return sequence, label
        return sequence

def collate_fn(batch):
    if len(batch[0]) == 2:
        sequences, labels = zip(*batch)
        sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
        labels = torch.stack(labels)
        return sequences, labels
    else:
        sequences = batch
        sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
        return sequences

In [7]:
class BiGRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, 
                 embedding_matrix=None, num_layers=2, dropout=0.3):
        super(BiGRUClassifier, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Initialize with pre-trained FastText embeddings if provided
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
            # Optionally freeze embeddings (uncomment to freeze)
            # self.embedding.weight.requires_grad = False
        
        self.gru = nn.GRU(
            embedding_dim, 
            hidden_dim, 
            num_layers=num_layers,
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Classification layer
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional
        
    def forward(self, x):
        # x shape: (batch_size, seq_len)
        
        # Embedding
        embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        
        # GRU (simpler than LSTM - only hidden state, no cell state)
        gru_out, hidden = self.gru(embedded)
        # gru_out shape: (batch_size, seq_len, hidden_dim * 2)
        # hidden shape: (num_layers * 2, batch_size, hidden_dim)
        
        # Use the last hidden state from both directions
        # For bidirectional GRU, we concatenate forward and backward hidden states
        forward_hidden = hidden[-2]  # Last layer, forward direction
        backward_hidden = hidden[-1]  # Last layer, backward direction
        final_hidden = torch.cat([forward_hidden, backward_hidden], dim=1)
        
        # Apply dropout
        final_hidden = self.dropout(final_hidden)
        
        # Classification
        output = self.classifier(final_hidden)  # (batch_size, num_classes)
        
        return output

# Model parameters
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3

# Initialize model
model = BiGRUClassifier(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=num_classes,
    embedding_matrix=embedding_matrix,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

print(f"Model initialized with {sum(p.numel() for p in model.parameters())} parameters")
print(model)

Model initialized with 6563730 parameters
BiGRUClassifier(
  (embedding): Embedding(19785, 300, padding_idx=0)
  (gru): GRU(300, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=6, bias=True)
)


In [8]:
BATCH_SIZE = 32

train_dataset = HateSpeechDataset(train_sequences, train_df['label_encoded'].tolist())
val_dataset = HateSpeechDataset(val_sequences, val_df['label_encoded'].tolist())
test_dataset = HateSpeechDataset(test_sequences)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Final test batches: {len(test_loader)}")

Training batches: 1111
Validation batches: 79
Final test batches: 79


In [9]:
# Utility functions for safe model saving/loading
def safe_save_checkpoint(checkpoint, filepath):
    """Save checkpoint with compatibility for different PyTorch versions"""
    try:
        torch.save(checkpoint, filepath)
    except Exception as e:
        print(f"Standard save failed: {e}")
        print("Trying with legacy serialization...")
        torch.save(checkpoint, filepath, _use_new_zipfile_serialization=False)

def safe_load_checkpoint(filepath):
    """Load checkpoint with compatibility for different PyTorch versions"""
    try:
        # Try default loading first
        return torch.load(filepath)
    except Exception as e:
        if "weights_only" in str(e) or "WeightsUnpickler" in str(e):
            print("Loading with weights_only=False due to sklearn objects in checkpoint...")
            return torch.load(filepath, weights_only=False)
        else:
            print(f"Loading error: {e}")
            print("Trying with map_location...")
            try:
                return torch.load(filepath, map_location=device)
            except:
                return torch.load(filepath, map_location=device, weights_only=False)

In [10]:
# Training configuration
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc="Training")
    for sequences, labels in progress_bar:
        sequences, labels = sequences.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        progress_bar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Acc': f'{100*correct/total:.2f}%'
        })
    
    return total_loss / len(train_loader), correct / total

# Validation function
def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for sequences, labels in tqdm(val_loader, desc="Validating"):
            sequences, labels = sequences.to(device), labels.to(device)
            
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_predictions)
    f1_macro = f1_score(all_labels, all_predictions, average='macro')
    f1_weighted = f1_score(all_labels, all_predictions, average='weighted')
    
    return total_loss / len(val_loader), accuracy, f1_macro, f1_weighted, all_predictions, all_labels

In [11]:
# Training loop
NUM_EPOCHS = 10
best_f1 = 0
best_model_path = "best_bigru_fasttext_model.pth"

print("Starting training with BiGRU + FastText...")
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 50)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc, val_f1_macro, val_f1_weighted, _, _ = validate(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step()
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print(f"Val F1 (macro): {val_f1_macro:.4f}, Val F1 (weighted): {val_f1_weighted:.4f}")
    print(f"Learning Rate: {scheduler.get_last_lr()[0]:.6f}")
    
    # Save best model
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro
        
        # Create checkpoint
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_f1': best_f1,
            'label_encoder': label_encoder,
            'word2idx': word2idx,
            'vocab': vocab
        }
        
        # Save with safe function
        safe_save_checkpoint(checkpoint, best_model_path)
        print(f"New best BiGRU model saved with F1: {best_f1:.4f}")

print(f"\nTraining completed! Best validation F1: {best_f1:.4f}")

Starting training with BiGRU + FastText...

Epoch 1/10
--------------------------------------------------


Validating: 100%|██████████| 79/79 [00:00<00:00, 1254.90it/s]


Train Loss: 0.9204, Train Acc: 0.6474
Val Loss: 0.8143, Val Acc: 0.6839
Val F1 (macro): 0.4254, Val F1 (weighted): 0.6453
Learning Rate: 0.001000
New best BiGRU model saved with F1: 0.4254

Epoch 2/10
--------------------------------------------------


Validating: 100%|██████████| 79/79 [00:00<00:00, 1239.69it/s]


Train Loss: 0.6836, Train Acc: 0.7358
Val Loss: 0.8107, Val Acc: 0.6839
Val F1 (macro): 0.4774, Val F1 (weighted): 0.6786
Learning Rate: 0.001000
New best BiGRU model saved with F1: 0.4774

Epoch 3/10
--------------------------------------------------


...

Train Loss: 0.0847, Train Acc: 0.9709
Val Loss: 2.0749, Val Acc: 0.6079
Val F1 (macro): 0.4209, Val F1 (weighted): 0.6079
Learning Rate: 0.000500

Epoch 9/10
--------------------------------------------------


Validating: 100%|██████████| 79/79 [00:00<00:00, 1249.21it/s]


Train Loss: 0.0781, Train Acc: 0.9731
Val Loss: 2.2333, Val Acc: 0.5983
Val F1 (macro): 0.4073, Val F1 (weighted): 0.6035
Learning Rate: 0.000500

Epoch 10/10
--------------------------------------------------


Validating: 100%|██████████| 79/79 [00:00<00:00, 1241.29it/s]


Train Loss: 0.0766, Train Acc: 0.9746
Val Loss: 2.3031, Val Acc: 0.6051
Val F1 (macro): 0.3916, Val F1 (weighted): 0.6030
Learning Rate: 0.000250

Training completed! Best validation F1: 0.4774


In [ ]:
checkpoint = safe_load_checkpoint(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print("Best BiGRU FastText model loaded for evaluation")

# Detailed validation evaluation
val_loss, val_acc, val_f1_macro, val_f1_weighted, val_predictions, val_labels = validate(
    model, val_loader, criterion, device
)

print(f"\nFinal Validation Results (BiGRU + FastText):")
print(f"Accuracy: {val_acc:.4f}")
print(f"F1 Score (Macro): {val_f1_macro:.4f}")
print(f"F1 Score (Weighted): {val_f1_weighted:.4f}")

# Classification report
print("\nDetailed Classification Report:")
print(classification_report(val_labels, val_predictions, 
                          target_names=label_encoder.classes_, digits=4))

Loading with weights_only=False due to sklearn objects in checkpoint...
Best BiGRU FastText model loaded for evaluation


Validating: 100%|██████████| 79/79 [00:00<00:00, 1164.39it/s]


Final Validation Results (BiGRU + FastText):
Accuracy: 0.6839
F1 Score (Macro): 0.4774
F1 Score (Weighted): 0.6786

Detailed Classification Report:
                precision    recall  f1-score   support

       Abusive     0.5059    0.4539    0.4785       564
          None     0.7772    0.8077    0.7922      1451
Political Hate     0.5305    0.5979    0.5622       291
       Profane     0.7114    0.6752    0.6928       157
Religious Hate     0.4762    0.2632    0.3390        38
        Sexism     0.0000    0.0000    0.0000        11

      accuracy                         0.6839      2512
     macro avg     0.5002    0.4663    0.4774      2512
  weighted avg     0.6756    0.6839    0.6786      2512



In [ ]:
def predict_on_test(model, test_loader, device):
    model.eval()
    all_predictions = []
    
    with torch.no_grad():
        for sequences in tqdm(test_loader, desc="Predicting"):
            sequences = sequences.to(device)
            outputs = model(sequences)
            _, predicted = torch.max(outputs.data, 1)
            all_predictions.extend(predicted.cpu().numpy())
    
    return all_predictions


print(f"Generating BiGRU predictions for {len(test_df)} dev-test samples...")
test_predictions = predict_on_test(model, test_loader, device)
test_labels = label_encoder.inverse_transform(test_predictions)

print(f"Generated {len(test_predictions)} predictions")

submission_df = test_df.copy()
submission_df['label'] = test_labels
submission_df['model'] = 'BiGRU-FastText'

submission_df[['id', 'label', 'model']].to_csv('subtask_1A_bigru_fasttext.tsv', 
                                               sep='\t', index=False)
print("BiGRU FastText predictions saved to subtask_1A_bigru_fasttext.tsv")